In [2]:
import argparse
import os
import warnings
import math
from pathlib import Path

import numpy as np
import pandas as pd

# warnings.filterwarnings("ignore")

In [ ]:
def load_soil_data(station_id, base_dir):
    """Load soil station data (.dat) and return a datetime‐indexed DataFrame."""
    file_path = Path(base_dir) / f"SM_{station_id}.dat"
    df = pd.read_csv(file_path, sep=",")
    df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%y %H:%M', errors='coerce')
    df = df.set_index('Date')
    df.columns = df.columns.str.strip()
    for col in ['SWC_5', 'SWC_10', 'SWC_20', 'SWC_50',
                'T_5', 'T_10', 'T_20', 'T_50', 'Ppt', 'Flag']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def load_met_data(station_id, base_dir):
    """Load MET station data (.dat) and return a datetime‐indexed DataFrame."""
    file_path = Path(base_dir) / f"MET_{station_id}.dat"
    df = pd.read_csv(
        file_path,
        parse_dates=['Date'],
        index_col='Date',
        date_parser=lambda x: pd.to_datetime(x, format='%m/%d/%y %H:%M', errors='coerce'),
        dtype={
            'Ppt': np.float64,
            'Tair': np.float64,
            'RH': np.float64,
            'Wind speed': np.float64,
            'Wind direction': np.float64,
            'Srad': np.float64
        }
    )
    df.columns = df.columns.str.strip()
    df = df.loc['2015-01-01 00:00:00':]
    for col in ['Ppt', 'Tair', 'RH', 'Wind speed', 'Wind direction', 'Srad']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def merge_raw_data(station_id, soil_base_dir, met_base_dir):
    """Merge soil and MET data, keeping MET precipitation when both exist."""
    df_soil = load_soil_data(station_id, soil_base_dir)
    df_met = load_met_data(station_id, met_base_dir)
    merged = pd.merge(df_soil, df_met,
                      how='inner',
                      left_index=True,
                      right_index=True,
                      suffixes=('_soil', '_met'))
    if 'Ppt_soil' in merged.columns and 'Ppt_met' in merged.columns:
        merged['Ppt'] = merged['Ppt_met']
        merged.drop(columns=['Ppt_soil', 'Ppt_met'], inplace=True)
    return merged


def save_merged_data(df, station_id, output_dir):
    """Save merged raw data to CSV; return the file path."""
    os.makedirs(output_dir, exist_ok=True)
    out_path = Path(output_dir) / f"raw_merged_station_{station_id}.csv"
    df.to_csv(out_path)
    return str(out_path)


def find_missing_data(df: pd.DataFrame) -> dict:
    """Return a dict of columns → list of timestamps where data is NaN."""
    missing = {}
    for col in df.columns:
        idx = df.index[df[col].isnull()]
        if not idx.empty:
            missing[col] = idx.tolist()
    return missing


def find_and_replace_wrong_data(df):
    """
    Identify values outside valid ranges, replace them with NaN,
    and return (corrected_df, dict of col → invalid timestamps).
    """
    wrong = {}
    dfc = df.copy()

    # Soil Moisture: between 0 and 0.6
    for col in ['SWC_5', 'SWC_10', 'SWC_20', 'SWC_50']:
        if col in dfc.columns:
            bad = dfc.index[(dfc[col] < 0) | (dfc[col] > 0.6)]
            if not bad.empty:
                wrong[col] = bad.tolist()
                dfc.loc[bad, col] = np.nan

    # Precipitation: non-negative
    if 'Ppt' in dfc.columns:
        bad = dfc.index[dfc['Ppt'] < 0]
        if not bad.empty:
            wrong['Ppt'] = bad.tolist()
            dfc.loc[bad, 'Ppt'] = np.nan

    # RH: between 0 and 100%
    if 'RH' in dfc.columns:
        bad = dfc.index[(dfc['RH'] < 0) | (dfc['RH'] > 100)]
        if not bad.empty:
            wrong['RH'] = bad.tolist()
            dfc.loc[bad, 'RH'] = np.nan

    # Wind speed: 0 to 25 m/s
    if 'Wind speed' in dfc.columns:
        bad = dfc.index[(dfc['Wind speed'] < 0) | (dfc['Wind speed'] > 25)]
        if not bad.empty:
            wrong['Wind speed'] = bad.tolist()
            dfc.loc[bad, 'Wind speed'] = np.nan

    # Wind direction: 0 to 360°
    if 'Wind direction' in dfc.columns:
        bad = dfc.index[(dfc['Wind direction'] < 0) | (dfc['Wind direction'] > 360)]
        if not bad.empty:
            wrong['Wind direction'] = bad.tolist()
            dfc.loc[bad, 'Wind direction'] = np.nan

    # Solar radiation: must be non-negative
    if 'Srad' in dfc.columns:
        bad = dfc.index[dfc['Srad'] < 0]
        if not bad.empty:
            wrong['Srad'] = bad.tolist()
            dfc.loc[bad, 'Srad'] = np.nan

    # Soil Temperature and Air Temperature: -30°C to 60°C
    for col in ['T_5', 'T_10', 'T_20', 'T_50', 'Tair']:
        if col in dfc.columns:
            bad = dfc.index[(dfc[col] < -30) | (dfc[col] > 60)]
            if not bad.empty:
                wrong[col] = bad.tolist()
                dfc.loc[bad, col] = np.nan

    return dfc, wrong


def combine_nan_lists(missing, wrong):
    """Merge the two dicts of timestamp lists and sort each list."""
    combined = {}
    for key in set(missing) | set(wrong):
        combined[key] = sorted(missing.get(key, []) + wrong.get(key, []))
    return combined


def group_consecutive_dates(dates: list, freq: pd.Timedelta) -> list:
    """Group timestamps that are within 1.5×freq of each other."""
    if not dates:
        return []
    groups, current = [], [dates[0]]
    for prev, curr in zip(dates, dates[1:]):
        # If the time difference is less than or equal to 1.5 times the expected frequency,
        # consider the dates as consecutive.
        if curr - prev <= freq * 1.5:
            current.append(curr)
        else:
            groups.append(current)
            current = [curr]
    groups.append(current)
    return groups


def create_missing_summary_df(info):
    """Build a summary DataFrame from the merged NaN/invalid timestamp info."""
    rows = []
    for param, ts_list in info.items():
        for grp in group_consecutive_dates(sorted(ts_list), pd.Timedelta(hours=1)):
            rows.append({
                "Parameter": param,
                "Start Timestamp": grp[0],
                "End Timestamp": grp[-1],
                "Number Missing": len(grp)
            })
    return pd.DataFrame(rows)

## Stationarity Test

For linear regression approaches, it is good to see if our data has stationarity. If not, we must visualize and figure out the trend/seasonal trend occuring and feature engineer our data to achieve stationarity.

On the other hand, this step does not matter too much when training neural networks, as they will take advantage of the patterns. It is important to keep time embedding features, spatial features, and station id for as much information as possible.

In [23]:
from statsmodels.tsa.stattools import adfuller

def adf_test_multiple_periods(df, col_name, periods=None):

    if periods is None:
        start_date = df.index.min()
        end_date = df.index.max()
        
        # default periods
        periods = [
            ('Last Month', end_date - pd.Timedelta(days=30), end_date),
            ('Last Year', end_date - pd.Timedelta(days=365), end_date),
            ('Total Period', start_date, end_date)
        ]
    
    for period_name, start, end in periods:
        period_data = df.loc[start:end, col_name].dropna()
        
        if len(period_data) < 2:
            print(f"\nNot enough data for {period_name} period")
            continue
            
        result = adfuller(period_data)
        
        print(f"\nADF Test Results for {col_name} - {period_name}")
        print(f'ADF Statistic: {result[0]:.4f}')
        print(f'p-value: {result[1]:.4f}')
        print('Critical Values:')
        for key, value in result[4].items():
            print(f'  {key}: {value:.4f}')
        
        # See if data is stationary or not
        if result[1] <= 0.05:
            print(f"Conclusion: Stationary (reject H0)")
        else:
            print(f"Conclusion: Non-stationary (fail to reject H0)")

In [12]:
base_dir = Path("../data-cleanup/cleaned_data")
file_path = base_dir / "Station1_cleaned_data.csv"

df = pd.read_csv(file_path, index_col=0, parse_dates=True)
df.drop('Flag', axis=1, inplace=True)

In [32]:
custom_periods = [
    ('Spring 2019', '2019-03-01', '2019-05-31'),
    ('Summer 2019', '2019-06-01', '2019-08-31'),
    ('Fall 2019', '2019-09-01', '2019-11-30'),
    ('Winter 2019', '2019-12-01', '2020-02-28'),
    ('2019', '2019-01-01', '2019-12-31'),
    ('Total Period', '2015-01-01', '2020-12-31')
]
for col in df.columns:
    
    adf_test_multiple_periods(df, str(col), periods=custom_periods)


ADF Test Results for SWC_5 - Spring 2019
ADF Statistic: -3.3988
p-value: 0.0110
Critical Values:
  1%: -3.4333
  5%: -2.8629
  10%: -2.5675
Conclusion: Stationary (reject H0)

ADF Test Results for SWC_5 - Summer 2019
ADF Statistic: -3.4199
p-value: 0.0103
Critical Values:
  1%: -3.4333
  5%: -2.8629
  10%: -2.5675
Conclusion: Stationary (reject H0)

ADF Test Results for SWC_5 - Fall 2019
ADF Statistic: -3.3457
p-value: 0.0130
Critical Values:
  1%: -3.4334
  5%: -2.8629
  10%: -2.5675
Conclusion: Stationary (reject H0)

ADF Test Results for SWC_5 - Winter 2019
ADF Statistic: -2.5132
p-value: 0.1123
Critical Values:
  1%: -3.4334
  5%: -2.8629
  10%: -2.5675
Conclusion: Non-stationary (fail to reject H0)

ADF Test Results for SWC_5 - 2019
ADF Statistic: -5.6764
p-value: 0.0000
Critical Values:
  1%: -3.4311
  5%: -2.8619
  10%: -2.5669
Conclusion: Stationary (reject H0)


KeyboardInterrupt: 

## Apply Feature Engineering to Data

- Adds features for time to capture the cyclical nature of our data
- Adds features for precipitation to capture potential causes to moisture change (can be updated based on other projects' findings)
- Modifies wind features to learnable x/y velocities

Potential Additions:
- Make sure precipitation features are NaN'd out if original feature is missing
- Concat dependnet features first

In [7]:
loc_dict = {1: (30.3989, -98.6105), 
                2: (30.4193, -98.8046), 
                3: (30.4421, -98.8427), 
                4: (30.4600, -98.9407), 
                5: (30.2454, -98.7059), 
                6: (30.2758, -98.7242)}

def engineer_features(df, station_id):
    print(f"Engineering features for station {station_id}")

    # Precipitation features
    #! Update these to be be NaN if there is a missing value in 'Ppt'
    df['Ppt_RainFlag'] = (df['Ppt'] > 0).astype(int)
    df['Ppt_3h_sum'] = df['Ppt'].rolling(3, min_periods=1).sum()
    df['Ppt_24h_sum'] = df['Ppt'].rolling(24, min_periods=1).sum()
    # time‑since‑rain:
    hours_since = []
    count = 0
    
    for v in df['Ppt']:
        count = 0 if v > 0 else count + 1
        hours_since.append(count)
    df['HoursSinceRain'] = hours_since

    # Wind features
    if 'Wind speed' in df and 'Wind direction' in df:
        wv = df['Wind speed']
        wd_rad = np.deg2rad(df['Wind direction']) 
        df['Wx'] = wv * np.cos(wd_rad)
        df['Wy'] = wv * np.sin(wd_rad)
    else:
        print(f"Skipping wind features for Station {station_id} (columns missing)")

    # Time features
    timestamp_s = df.index.map(pd.Timestamp.timestamp)
    day, year, month = 86400, 86400 * 365.2425, 86400 * 30.4167
    df['DaySin'] = np.sin(timestamp_s * (2 * np.pi / day))
    df['DayCos'] = np.cos(timestamp_s * (2 * np.pi / day))
    df['YearSin'] = np.sin(timestamp_s * (2 * np.pi / year))
    df['YearCos'] = np.cos(timestamp_s * (2 * np.pi / year))
    df['MonthSin'] = np.sin(timestamp_s * (2 * np.pi / month))
    df['MonthCos'] = np.cos(timestamp_s * (2 * np.pi / month))
    
    # Spatial features
    df['latitude'], df['longitude'] = loc_dict[station_id]

    # Station ID hot-encoding
    df['station_id'] = station_id

    return df

In [8]:
base_dir = Path("../data-cleanup/cleaned_data")

for station_id in range (1, 7):
    feature_engineered_out = f"feature_engineered/Station{station_id}_engineered_data.csv"
    os.makedirs(Path(feature_engineered_out).parent, exist_ok=True)
    
    file_path = base_dir / f"Station{station_id}_cleaned_data.csv"
    
    df = pd.read_csv(file_path, index_col=0, parse_dates=True)
    
    df = engineer_features(df, station_id)
    
    df.drop('Flag', axis=1, inplace=True)
    
    #! Optional hot-encoding of station_id to potentially aid deep-learning network
    df['station_id'] = station_id
    
    df.to_csv(feature_engineered_out, na_rep="NaN", index=True, index_label='datetime')
    print(f"Feature Engineered full‐timeline data for Station {station_id} saved to: {feature_engineered_out}")

Engineering features for station 1
Feature Engineered full‐timeline data for Station 1 saved to: feature_engineered/Station1_engineered_data.csv
Engineering features for station 2
Feature Engineered full‐timeline data for Station 2 saved to: feature_engineered/Station2_engineered_data.csv
Engineering features for station 3
Feature Engineered full‐timeline data for Station 3 saved to: feature_engineered/Station3_engineered_data.csv
Engineering features for station 4
Feature Engineered full‐timeline data for Station 4 saved to: feature_engineered/Station4_engineered_data.csv
Engineering features for station 5
Feature Engineered full‐timeline data for Station 5 saved to: feature_engineered/Station5_engineered_data.csv
Engineering features for station 6
Feature Engineered full‐timeline data for Station 6 saved to: feature_engineered/Station6_engineered_data.csv


## Data Pre-processing

In [9]:
from sklearn.preprocessing import StandardScaler

In [10]:
# Normalize features
def normalize_features(df, features):
    scaler = StandardScaler()

    df[features] = scaler.fit_transform(df[features])
        
    return df, scaler

In [29]:
base_dir = Path("../data-cleanup/feature_engineered")
file_path = base_dir / f"Station1_engineered_data.csv"

df = pd.read_csv(file_path)


In [30]:
exclude_cols = ['latitude', 'longitude', 'Ppt_RainFlag', 'Ppt_3h_sum', 'Ppt_24h_sum', 'HoursSinceRain', "Wind speed", "Wind direction", "datetime"]
features = [col for col in df.columns if col not in exclude_cols and ('Sin' not in col and 'Cos' not in col)]

df, scaler = normalize_features(df, features)

lat_sum, long_sum, lat_sq_sum, long_sq_sum = 0, 0, 0, 0
for station_id in range(1, 7):
    base_dir = Path("../data-cleanup/feature_engineered")
    file_path = base_dir / f"Station{station_id}_engineered_data.csv"

    df_temp = pd.read_csv(file_path)
    
    lat_sum += df_temp['latitude'][0]
    long_sum += df_temp['longitude'][0]
    lat_sq_sum += df_temp['latitude'][0]**2
    long_sq_sum += df_temp['longitude'][0]**2
    
lat_mean, long_mean = lat_sum/6, long_sum/6
lat_std, long_std = lat_sq_sum / 6 - lat_mean**2, long_sq_sum / 6 - long_mean**2

df['latitude'] = (df['latitude'] - lat_mean) / lat_std
df['longitude'] = (df['longitude'] - long_mean) / long_std
    

In [ ]:
print(df)

## Window Creation

Todo:

Create 3 different types of window/label schemes
- X: Artifically masked window; y: Original window
- X: Past Window; y: Present Values
- X: Past and Future window; y: Present Values

In [36]:
import torch
from numpy.lib.stride_tricks import sliding_window_view

In [37]:
def make_windows(df, window_size=24, overlap = 0.5):
    
    if not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame")
    if window_size <= 0:
        raise ValueError("window_size must be positive")
    if not 0 <= overlap < 1:
        raise ValueError("overlap must be between 0 and 1")
    if len(df) == 0:
        raise ValueError("df cannot be empty")
    
    df_numeric = df.drop('datetime', axis=1)
    data = df_numeric.values
  
    step_size = int(window_size * (1 - overlap))
    windows = sliding_window_view(data, window_size, axis=0)[::step_size]
    
    valid_windows = ~np.isnan(windows).any(axis=(1, 2))
    
    return torch.tensor(windows[valid_windows], dtype=torch.float32).transpose(1, 2)
        

## Training / Testing Split

In [33]:
from torch.utils.data import TensorDataset, DataLoader, random_split

In [34]:
def load_data (windows_data, batch_size = 32, val_split = 0.1):
    
    X = windows_data #masked

    # Labels is a copy of X because we are trying to predict original window
    y = X.clone()

    dataset = TensorDataset(X, y)
    
    val_size = int(val_split * len(dataset))
    train_size = len(dataset) - val_size
    
    train_dataset, val_dataset = random_split(
        dataset, [train_size, val_size]
    )
    
    train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=8,
            pin_memory=True
        )
        
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=8,
        pin_memory=True
    )
    
    print(f"Dataset loaded with {len(dataset)} windows")
    print(f"Training samples: {train_size}, Validation samples: {val_size}")
    
    return train_loader, val_loader


In [43]:
window_size = 24
overlap = 0.5
base_dir = Path("../data-cleanup/feature_engineered")

all_station_windows = []
    
for station_id in range (1, 7):
    file_path = base_dir / f"Station{station_id}_engineered_data.csv"
    df = pd.read_csv(file_path)
    
    windows = make_windows(df, window_size, overlap)

    all_station_windows.append(windows)
    
windows_data = torch.cat(all_station_windows, dim=0)

# train_loader, val_loader = load_data(windows_data)

### Masking

Potential Additions:
- Mask out dependent features from the original features (Ex: if 'Ppt' is masked out, mask out 'Ppt_3h_sum')


In [85]:
ORIGINAL_FEATURE_NUM = 20

def create_random_mask(data, mask_amt = 0.1):
    
    num_windows, window_size, num_features = data.shape
    vals_per_window = window_size * ORIGINAL_FEATURE_NUM
    num_vals_mask = int(vals_per_window * mask_amt)
    
    # Produce a random distribution in the shape of (num_windows, values per window)
    rand = torch.rand((num_windows, vals_per_window))
    
    # Select the smallest 'num_vals_mask' random values and grabs the largest one as the threshold
    thresholds = torch.topk(rand, num_vals_mask, largest=False, dim=1).values[:, -1].unsqueeze(1)
    
    # Makes a flat mask where True are the 'num_vals_mask' random values that are below the threshold
    mask_flat = rand <= thresholds
    
    # Reshape mask to original data shape
    mask = torch.zeros_like(data, dtype=torch.bool)
    
    actual_mask = mask_flat.view(num_windows, window_size, ORIGINAL_FEATURE_NUM)
    
    mask[:,:window_size,:ORIGINAL_FEATURE_NUM] = actual_mask
    
    return mask.bool()

def create_block_mask(data, mask_amt = 0.1, max_block_size = 0):
    
    num_windows, row_range, _ = data.shape
    col_range = ORIGINAL_FEATURE_NUM
    vals_per_window = row_range * col_range
    num_vals_mask = int(vals_per_window * mask_amt)
    
    mask = torch.zeros_like(data, dtype=torch.bool)
    
    for i in range(num_windows):
        masked_vals = 0
        while num_vals_mask > masked_vals:
            remaining_vals = num_vals_mask - masked_vals
            
            max_possible_block = min(remaining_vals, row_range * col_range)
            if max_block_size > 0:
                max_possible_block = min(max_possible_block, max_block_size)
            
            block_width = torch.randint(1, min(max_possible_block, row_range) + 1, (1,)).item()
            block_len = torch.randint(1, min(max_possible_block // block_width, col_range) + 1, (1,)).item()
                
            r = torch.randint(0, row_range - block_width + 1, (1,)).item()
            c = torch.randint(0, col_range - block_len + 1, (1,)).item()
            
            masked_before = mask[i, r:r+block_width, c:c+block_len].sum().item()
            
            mask[i, r:r+block_width, c:c+block_len] = True       
            
            newly_masked = block_width * block_len - masked_before
            masked_vals += newly_masked

    return mask.bool()

def create_row_mask(data, mask_amt = 0.1):
    
    num_windows, row_range, _ = data.shape
    col_range = ORIGINAL_FEATURE_NUM
    vals_per_window = row_range * col_range
    num_vals_mask = int(vals_per_window * mask_amt)

    mask = torch.zeros_like(data, dtype=torch.bool)
    
    for i in range(num_windows):
        masked_vals = 0
        while num_vals_mask > masked_vals:
            remaining_vals = num_vals_mask - masked_vals
            
            max_possible_block = min(remaining_vals, row_range * col_range)
            
            block_len = col_range
            block_width = torch.randint(1, min(math.ceil(max_possible_block / block_len), row_range) + 1, (1,)).item()
            
            r = torch.randint(0, row_range - block_width + 1, (1,)).item()

            masked_before = mask[i, r:r+block_width,:col_range].sum().item()
            # masked_before = torch.isnan(data[i][r: r + block_width]).sum().item()

            mask[i, r: r + block_width,:col_range] = True   
            # data[i][r: r + block_width] = torch.nan
            
            # print("Expected shape to mask:", (block_width, block_len))
            # print("Got:", data[i][r: r + block_width].shape)
            
            newly_masked = block_width * block_len - masked_before
            masked_vals += newly_masked
        
        
    return mask.bool()

def create_col_mask(data, mask_amt = 0.1):
  
    num_windows, row_range, _ = data.shape
    col_range = ORIGINAL_FEATURE_NUM
    vals_per_window = row_range * col_range
    num_vals_mask = int(vals_per_window * mask_amt)
    
    mask = torch.zeros_like(data, dtype=torch.bool)
    
    for i in range(num_windows):
        masked_vals = 0
        while num_vals_mask > masked_vals:
            remaining_vals = num_vals_mask - masked_vals
            
            max_possible_block = min(remaining_vals, row_range * col_range)
            
            block_width = row_range
            block_len = torch.randint(1, min(math.ceil(max_possible_block / block_width), col_range) + 1, (1,)).item()
            
            c = torch.randint(0, col_range - block_len + 1, (1,)).item()

            masked_before = mask[i,:, c:c+block_len].sum().item()
            # masked_before = torch.isnan(data[i][:, c:c+block_len]).sum().item()
            
            mask[i,:, c:c+block_len] = True   
            # data[i][:, c:c+block_len] = torch.nan
            
            # print("Expected shape to mask:", (block_width, block_len))
            # print("Got:", mask[i,:, c:c+block_len].shape)
            
            newly_masked = block_width * block_len - masked_before
            masked_vals += newly_masked

    return mask.bool()

In [86]:
tot_mask_amt = 0.1

mask_amts = np.random.dirichlet(np.ones(4)) * tot_mask_amt

print(mask_amts)

rand_mask = create_random_mask(windows_data, mask_amts[0])
block_mask = create_block_mask(windows_data, mask_amts[1])
row_mask = create_row_mask(windows_data, mask_amts[2])
col_mask = create_col_mask(windows_data, mask_amts[3])

final_mask = rand_mask | block_mask | row_mask | col_mask

windows_data_masked = windows_data.clone()
windows_data_masked[final_mask] = torch.nan

[0.03141275 0.00244566 0.03669109 0.02945049]


In [87]:
df = pd.DataFrame(windows_data_masked[0])

In [ ]:
print(df)